# Module 1 — Brief 2 — Itération et amélioration du modèle

Ce notebook couvre les étapes du Brief 2 :

- Relecture du diagnostic du Brief 1
- Mise en oeuvre des actions correctives (dataset et/ou hyperparamètres)
- Réentraînement — Run 2
- Comparaison Run 1 vs Run 2 dans MLflow
- Choix et sauvegarde du meilleur modèle

Les cellules marquées `### A COMPLÉTER ###` contiennent du code partiel à terminer.
Les cellules sans marqueur sont fournies et peuvent être exécutées directement.

Prérequis : le notebook du Brief 1 a été exécuté, le Run 1 est disponible dans MLflow.

---
## 0. Imports et configuration

In [2]:
import json
import torch
import mlflow
from collections import Counter
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

print("Imports OK")

d:\Brice\Simplon_formation_ia\M1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device utilisé : {device}")

Device utilisé : cuda


In [3]:
MODEL_ID = "meta-llama/Llama-3.2-1B"
MAX_LENGTH = 512

CATEGORIES_VALIDES = [
    "Support technique",
    "Demande commerciale",
    "Demande de transformation",
    "Réclamation",
    "Information générale"
]

---
## 1. Relecture du diagnostic du Brief 1

Avant toute action, relisez votre diagnostic du Brief 1.
Les actions de ce notebook doivent être directement motivées par ce diagnostic.
Une action corrective sans diagnostic est une intuition, pas une démarche.

In [ ]:
### A COMPLÉTER ###
# Objectif : résumer le diagnostic du Brief 1 avant d'agir
#
# Répondez aux questions suivantes dans des commentaires :
#
# 1. Quel était l'écart train_loss / eval_loss du Run 1 ?
# 2. Quelles catégories étaient mal classifiées ?
# 3. Le JSON généré était-il systématiquement valide ?
# 4. Quelle(s) action(s) corrective(s) avez-vous décidé d'appliquer ?
#    (Option A : dataset, Option B : hyperparamètres, Option C : les deux)

# 1. Ecart train_loss / eval_loss :
#   L'écart est de 1.1636 (1.7566 − 0.5930) sur le run 1.

# 2. Catégories problématiques :
#   toutes les catégories.

# 3. Qualité du JSON :
#   le modèle ne génère pas de JSON valide

# 4. Action(s) corrective(s) choisie(s) et justification :
#   Ajouter des exemples spécifiques au dataset pour améliorer la compréhension des catégories et des priorités
#   Augmentation du nombre d'epochs, passer de 3 à 4 epochs pour permettre au modèle d'apprendre plus en profondeur les exemples du dataset.
#   Améliorer le prompt pour mieux guider le modèle vers la structure JSON attendue et les valeurs spécifiques des catégories et priorités.
# Sur la loss : Je m'attends à ce que la train_loss descende de manière régulière et se stabilise à un niveau bas, 
#   proche de la eval_loss.
#   Sur les prédictions : un respect du format JSON, des catégories conformes à la liste prédéfinie, et des priorités correctes."

---
## 2. Action corrective — Option A : compléter le dataset

Si votre diagnostic a révélé que certaines catégories sont mal classifiées,
le dataset est probablement insuffisant sur ces cas.

Ajoutez des exemples ciblés sur les catégories ou types de demandes problématiques.

Si vous avez choisi l'Option B uniquement, passez directement à la section 3.

In [5]:
# Chargement du dataset initial
with open("dataset_fastia_module1.jsonl", "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

print(f"Dataset initial : {len(raw_data)} exemples")
print("\nDistribution initiale :")
for cat, count in Counter(e["output"]["categorie"] for e in raw_data).items():
    print(f"  {cat} : {count}")

Dataset initial : 86 exemples

Distribution initiale :
  Support technique : 20
  Demande commerciale : 16
  Demande de transformation : 15
  Réclamation : 15
  Information générale : 20


In [6]:
### A COMPLÉTER — Option A uniquement ###
# Objectif : ajouter des exemples ciblés sur les catégories problématiques
#
# 1. Identifier les catégories à renforcer d'après votre diagnostic
# 2. Rédiger entre 20 et 50 nouveaux exemples en respectant le format JSON du dataset
# 3. Chaque exemple doit avoir la structure :
#    { "input": "...", "output": { "categorie": "...", "priorite": "...", "reponse_suggeree": "..." } }
#
# Consignes de rédaction :
# - Ne pas copier des exemples existants
# - Varier le style : formel, informel, court, long
# - Couvrir des cas limites ou ambigus si vous en avez observés

nouveaux_exemples = [
  { "input": "L'option 'Mode Sombre' fait ramer toute l'interface sur mon navigateur Brave, c'est super lent.", "output": { "categorie": "Support technique", "priorite": "normale", "reponse_suggeree": "Nous avons noté un problème d'optimisation spécifique au moteur Chromium sur Brave. Notre équipe front-end travaille sur un correctif de performance." } },
  { "input": "URGENT : On n'arrive plus à générer de clés API depuis le dashboard, le bouton est grisé.", "output": { "categorie": "Support technique", "priorite": "haute", "reponse_suggeree": "Un incident sur le service d'authentification a été détecté. Nos ingénieurs sont en train de rétablir les droits de génération de clés." } },
  { "input": "Est-ce que vous supportez l'import de schémas GraphQL pour la configuration de l'IA ?", "output": { "categorie": "Support technique", "priorite": "normale", "reponse_suggeree": "L'import GraphQL est supporté via notre module de configuration avancée. Je vous envoie la procédure technique par email." } },
  { "input": "Ma session se déconnecte toutes les 5 minutes, c'est impossible de travailler !", "output": { "categorie": "Support technique", "priorite": "normale", "reponse_suggeree": "Cela semble lié à un conflit de cookies ou de politique de sécurité locale. Nous allons vérifier vos logs de connexion pour identifier la cause." } },
  { "input": "Le plugin pour Visual Studio Code ne reconnaît pas mes identifiants FastIA ce matin.", "output": { "categorie": "Support technique", "priorite": "normale", "reponse_suggeree": "Une mise à jour du SDK a été déployée cette nuit. Veuillez mettre à jour votre extension vers la version 2.4.1 pour résoudre le problème." } },

  { "input": "On aimerait bien un tarif de groupe si on prend 200 licences d'un coup, vous faites ça ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "Absolument, nous proposons des tarifs dégressifs pour les volumes importants. Un conseiller va vous transmettre notre grille tarifaire 'Grand Compte'." } },
  { "input": "Bonjour, est-ce que vos solutions sont éligibles au crédit impôt recherche (CIR) ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "Nos prestations de R&D en IA sont effectivement éligibles au CIR. Nous pouvons vous fournir les agréments nécessaires pour votre dossier." } },
  { "input": "On hésite avec votre concurrent direct, quels sont vos avantages majeurs sur la partie NLP ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "Notre moteur NLP se distingue par sa gestion des dialectes spécifiques et son déploiement on-premise. Organisons un comparatif technique cette semaine." } },
  { "input": "Je cherche à acheter le pack 'Expert' mais le lien de paiement est mort.", "output": { "categorie": "Demande commerciale", "priorite": "haute", "reponse_suggeree": "Toutes nos excuses. Notre équipe commerciale vous contacte immédiatement pour finaliser la transaction via une procédure sécurisée alternative." } },
  { "input": "Est-il possible d'avoir un contrat de support 24/7 avec un interlocuteur dédié ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "Cette option est disponible dans notre offre 'Platinum'. Nous vous envoyons le descriptif des services de support étendu dès maintenant." } },
  { "input": "Hello ! On lance une asso à but non lucratif, vous avez des réductions pour nous ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "FastIA soutient les initiatives associatives avec des remises allant jusqu'à 50%. Envoyez-nous vos statuts pour débloquer votre tarif spécial." } },
  { "input": "On a besoin d'intégrer vos API dans notre application SaaS d'ici 48h, qui peut nous aider ?", "output": { "categorie": "Demande commerciale", "priorite": "haute", "reponse_suggeree": "Notre équipe Solutions Architects peut vous accompagner en urgence. Un expert vous contacte pour un sprint d'intégration rapide." } },
  { "input": "Serait-il possible de tester la version Enterprise pendant 15 jours sur nos propres données ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "Nous proposons effectivement des POC (Proof of Concept) encadrés. Nous allons mettre en place un environnement de test sécurisé pour vos données." } },
  { "input": "Quelles sont les modalités de résiliation si on prend un engagement annuel ?", "output": { "categorie": "Demande commerciale", "priorite": "normale", "reponse_suggeree": "Nos contrats annuels prévoient un préavis de 3 mois. Je vous joins un exemplaire de nos conditions générales de vente pour plus de détails." } },

  { "input": "On veut passer d'un système papier à une IA qui scanne nos factures automatiquement. Par quoi on commence ?", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "C'est un excellent cas d'usage. Nous recommandons de commencer par une phase de numérisation test avec notre module OCR pour évaluer la précision." } },
  { "input": "Notre boîte veut devenir 'AI-First' d'ici 2027. Vous faites des diagnostics de culture d'entreprise ?", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "Le changement culturel est la clé. Nous proposons des audits d'acculturation IA pour identifier les freins internes et former vos leaders." } },
  { "input": "On a 10 To de données pas du tout structurées, on peut en tirer quelque chose avec vous ?", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "Vos 'dark data' sont une mine d'or. Nous suggérons un atelier de Data Discovery pour structurer ces informations et définir des cas d'usage." } },
  { "input": "Besoin d'un plan de transition pour migrer nos anciens algorithmes vers des modèles de type LLM.", "output": { "categorie": "Demande de transformation", "priorite": "haute", "reponse_suggeree": "La modernisation vers les modèles génératifs demande une approche par étapes. Nos experts LLM vont analyser vos anciens scripts pour définir le plan de migration." } },
  { "input": "On veut automatiser la détection de défauts sur notre chaîne de montage avec de la vision par ordinateur.", "output": { "categorie": "Demande de transformation", "priorite": "haute", "reponse_suggeree": "C'est une transformation industrielle majeure. Nous devons dépêcher un expert sur site pour évaluer vos caméras et l'infrastructure réseau nécessaire." } },
  { "input": "Comment peut-on utiliser l'IA pour réduire notre consommation d'énergie dans nos entrepôts ?", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "Le 'Smart Building' via IA est très efficace. Nous pouvons installer des capteurs pour optimiser vos flux et réduire votre empreinte carbone de 20%." } },
  { "input": "On souhaite créer un chatbot interne pour que nos employés accèdent aux procédures RH facilement.", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "L'automatisation du support interne est un projet rapide à forte valeur. Nous pouvons déployer une version bêta connectée à votre base de connaissances en 15 jours." } },
  { "input": "Est-ce que vous pouvez nous aider à monter un centre d'excellence IA en interne ?", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "L'internalisation des compétences est primordiale. Nous vous accompagnons dans le recrutement et la structuration de votre futur 'AI Lab'." } },
  { "input": "On doit revoir toute notre cybersécurité pour intégrer des outils IA de détection de menaces.", "output": { "categorie": "Demande de transformation", "priorite": "haute", "reponse_suggeree": "L'IA défensive est indispensable aujourd'hui. Notre consultant SecOps vous contacte pour un audit de vulnérabilité de votre SI actuel." } },
  { "input": "Peut-on utiliser l'IA générative pour aider nos designers à créer des maquettes plus vite ?", "output": { "categorie": "Demande de transformation", "priorite": "normale", "reponse_suggeree": "Oui, nous proposons des outils de design assisté par IA. Un atelier de démonstration avec votre équipe créative permettrait d'en voir les bénéfices." } },

  { "input": "Vous m'avez prélevé deux fois ce mois-ci, c'est quoi ce délire ?", "output": { "categorie": "Réclamation", "priorite": "haute", "reponse_suggeree": "Nous regrettons sincèrement cet incident technique. Notre service comptabilité procède au remboursement immédiat du second prélèvement." } },
  { "input": "Le consultant qui devait venir aujourd'hui n'est jamais venu et ne répond pas au téléphone !", "output": { "categorie": "Réclamation", "priorite": "haute", "reponse_suggeree": "C'est inadmissible et nous vous présentons nos excuses. Nous vérifions l'urgence et vous envoyons un remplaçant ou planifions un dédommagement." } },
  { "input": "Votre interface est devenue super complexe après la mise à jour, on ne s'y retrouve plus du tout.", "output": { "categorie": "Réclamation", "priorite": "normale", "reponse_suggeree": "Vos retours sur l'UX sont essentiels. Nous organisons un point avec notre équipe produit pour simplifier les menus qui vous posent problème." } },
  { "input": "J'ai demandé une clôture de compte il y a un mois et je reçois encore des relances de paiement !", "output": { "categorie": "Réclamation", "priorite": "haute", "reponse_suggeree": "Il semble y avoir une désynchronisation dans notre système de résiliation. Nous stoppons immédiatement les relances et confirmons la clôture." } },
  { "input": "Les réponses de votre IA sont devenues incohérentes et pleines d'erreurs depuis deux jours.", "output": { "categorie": "Réclamation", "priorite": "haute", "reponse_suggeree": "Nous avons détecté une dérive sur l'un de nos modèles suite à une mise à jour. Nous effectuons un rollback immédiat vers la version stable." } },
  { "input": "Vous aviez promis une compatibilité avec Outlook pour mars, on est en juin et toujours rien.", "output": { "categorie": "Réclamation", "priorite": "normale", "reponse_suggeree": "Nous comprenons votre déception. Le développement a pris du retard mais la bêta est prête. Nous vous offrons un accès anticipé pour vous remercier de votre patience." } },
  { "input": "Votre support technique me répond avec des messages automatiques qui ne servent à rien.", "output": { "categorie": "Réclamation", "priorite": "normale", "reponse_suggeree": "Nous sommes désolés pour cette expérience frustrante. Je transfère personnellement votre dossier à un technicien senior pour une aide personnalisée." } },
  { "input": "L'application mobile n'arrête pas de crasher sur mon téléphone, c'est inutilisable professionnellement.", "output": { "categorie": "Réclamation", "priorite": "haute", "reponse_suggeree": "La stabilité mobile est notre priorité. Nous avons besoin de votre modèle de téléphone pour pousser un correctif urgent sur votre version." } },
  { "input": "Le devis que j'ai reçu est 30% plus cher que ce qu'on avait discuté oralement, c'est pas sérieux.", "output": { "categorie": "Réclamation", "priorite": "normale", "reponse_suggeree": "Une erreur de saisie a pu se glisser dans la proposition. Notre directeur commercial va reprendre les termes de votre projet avec vous aujourd'hui." } },
  { "input": "On a perdu des données clients à cause d'un bug sur votre module d'import, on est furieux.", "output": { "categorie": "Réclamation", "priorite": "haute", "reponse_suggeree": "Cette situation est critique. Notre équipe de data recovery est mobilisée pour tenter de restaurer vos données depuis nos sauvegardes de sécurité." } },

  { "input": "Est-ce que vous publiez un rapport annuel sur l'impact carbone de vos serveurs ?", "output": { "categorie": "Information générale", "priorite": "normale", "reponse_suggeree": "Oui, notre rapport RSE incluant notre bilan carbone numérique est public et disponible dans la section 'À propos' de notre site." } },
  { "input": "Quels sont les horaires d'ouverture de votre accueil téléphonique ?", "output": { "categorie": "Information générale", "priorite": "normale", "reponse_suggeree": "Notre accueil est ouvert du lundi au vendredi, de 8h30 à 18h30 sans interruption." } },
  { "input": "Vous avez des bureaux à Lyon ou c'est seulement sur Paris ?", "output": { "categorie": "Information générale", "priorite": "normale", "reponse_suggeree": "Notre siège est à Paris, mais nous avons également des bureaux à Lyon (quartier Part-Dieu) et Nantes." } },
  { "input": "Est-ce que vos équipes interviennent pour des conférences lors de salons professionnels ?", "output": { "categorie": "Information générale", "priorite": "normale", "reponse_suggeree": "Nos experts participent régulièrement à des conférences IA. Contactez notre service communication pour toute demande d'intervention." } },
  { "input": "Où peut-on trouver la liste des postes ouverts chez FastIA en ce moment ?", "output": { "categorie": "Information générale", "priorite": "normale", "reponse_suggeree": "Toutes nos offres d'emploi sont listées sur notre page 'Carrières' ainsi que sur notre profil LinkedIn." } }
]

print(f"Nouveaux exemples rédigés : {len(nouveaux_exemples)}")

Nouveaux exemples rédigés : 39


In [ ]:
### A COMPLÉTER — Option A uniquement ###
# Objectif : fusionner le dataset initial avec les nouveaux exemples
#
# 1. Fusionner raw_data et nouveaux_exemples
# 2. Afficher la nouvelle taille du dataset
# 3. Afficher la nouvelle distribution des catégories
# 4. Vérifier que la distribution est plus équilibrée qu'avant
#
# 5. Sauvegarder le dataset enrichi dans dataset_fastia_v2.json
#
# Question : la distribution est-elle suffisamment équilibrée ?
# Un déséquilibre résiduel peut-il encore poser problème ?
# Répondez dans un commentaire.

raw_data_v2 = raw_data + nouveaux_exemples

print(f"Dataset enrichi : {len(raw_data_v2)} exemples")
print("\nNouvelle distribution :")
for cat, count in Counter(exemple["output"]["categorie"] for exemple in raw_data_v2).items():
    print(f"  {cat} : {count} exemples")

with open("dataset_fastia_v2.jsonl", "w", encoding="utf-8") as f:
    for exemple in raw_data_v2:
        json.dump(exemple, f, ensure_ascii=False)
        f.write("\n")

print("\nDataset sauvegardé dans dataset_fastia_v2.jsonl")

# Votre réponse :
# la distribution est-elle suffisamment équilibrée ?
#   La distribution est uniforme avec 25 exemples par catégorie.

# Un déséquilibre résiduel peut-il encore poser problème ?
#   il peut y avoir un désiquilibre sr la diversité des formulations ou des cas d'usage au sein de chaque catégorie, 
#   même si le nombre d'exemples est égal. 


Dataset enrichi : 125 exemples

Nouvelle distribution :
  Support technique : 25 exemples
  Demande commerciale : 25 exemples
  Demande de transformation : 25 exemples
  Réclamation : 25 exemples
  Information générale : 25 exemples

Dataset sauvegardé dans dataset_fastia_v2.jsonl


---
## 3. Action corrective — Option B : ajuster les hyperparamètres

Si votre diagnostic a révélé un sous-apprentissage ou un sur-apprentissage,
ajustez les hyperparamètres en conséquence.

Rappel des signaux :
- Sous-apprentissage : loss élevée, prédictions incohérentes
  -> augmenter le learning rate ou le nombre d'époques
- Sur-apprentissage : train loss basse, eval loss haute
  -> réduire le learning rate, réduire les époques, augmenter lora_dropout

Si vous avez choisi l'Option A uniquement, conservez les hyperparamètres du Run 1.

In [16]:
### A COMPLÉTER ###
# Objectif : définir les hyperparamètres du Run 2
#
# 1. Définir RUN2_CONFIG avec les valeurs ajustées
# 2. Pour chaque valeur modifiée, expliquer pourquoi dans un commentaire
#
# Si Option A uniquement : recopier les valeurs du Run 1 sans les modifier

RUN1_CONFIG = {
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05
}

RUN2_CONFIG = {
    "learning_rate": 2e-4,
    "num_train_epochs": 4,
    "per_device_train_batch_size": 4,
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "run_name": "run2"
}

# Justification des changements :
#   Passage de 3 à 4 époques pour permettre au modèle une meilleure compréhension.

---
## 4. Préparation du dataset pour le Run 2

On reconstruit le pipeline de préparation des données avec le dataset
approprié selon l'action corrective choisie.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer chargé")

Tokenizer chargé


In [10]:
def build_prompt(exemple):
    instruction = (
        "Rôle : Tu es un expert en classification de tickets support pour FastIA."
        "Mission : Analyse la demande utilisateur et renvoie exclusivement un objet JSON."
        "Contraintes strictes :"
        "Format : Réponds uniquement en JSON pur, sans texte avant ou après (pas de \"Voici le résultat\")."
        "Champs obligatoires : categorie, priorite, reponse_suggeree."
        "Catégories autorisées (Choisir exclusivement parmi) : "
        "Support technique"
        "Demande commerciale"
        "Demande de transformation"
        "Réclamation"
        "Information générale"
        "Priorités autorisées : Moyenne, Haute."
        "Langue : Tout le contenu du JSON doit être en Français.\n\n"
        f"Demande : {exemple['input']}"
    )
    reponse = json.dumps(exemple["output"], ensure_ascii=False)
    return f"<s>[INST] {instruction} [/INST] {reponse} </s>"


def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

In [13]:
### A COMPLÉTER ###
# Objectif : préparer le dataset pour le Run 2
#
# 1. Charger le bon dataset selon votre action corrective :
#    - Option A ou C : charger dataset_fastia_v2.json
#    - Option B uniquement : charger dataset_fastia_module1.json
#
# 2. Appliquer build_prompt à tous les exemples
# 3. Tokeniser avec .map()
# 4. Diviser en train (80%) et test (20%)
# 5. Afficher la taille des deux jeux

dataset_path = "dataset_fastia_v2.jsonl" 

with open(dataset_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

prompts = [build_prompt(exemple) for exemple in data]
dataset = Dataset.from_dict({"text": prompts})
dataset_tokenise = dataset.map(lambda x: tokenize(x['text']), batched=True)
split = dataset_tokenise.train_test_split(test_size=0.2)

train_dataset = split["train"]
test_dataset = split["test"]

print(f"Entraînement : {len(train_dataset)} exemples")
print(f"Test         : {len(test_dataset)} exemples")

Map: 100%|██████████| 125/125 [00:00<00:00, 1096.08 examples/s]

Entraînement : 100 exemples
Test         : 25 exemples


---
## 5. Chargement du modèle et configuration LoRA — Run 2

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Modèle de base chargé")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:03<00:00, 45.59it/s]


Modèle de base chargé


In [17]:
### A COMPLÉTER ###
# Objectif : configurer LoRA avec les paramètres du Run 2
#
# Utiliser les valeurs de RUN2_CONFIG pour lora_r, lora_alpha, lora_dropout

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=RUN2_CONFIG["lora_r"],
    lora_alpha=RUN2_CONFIG["lora_alpha"],
    target_modules=["q_proj", "v_proj"],
    lora_dropout=RUN2_CONFIG["lora_dropout"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


---
## 6. Entraînement — Run 2

In [18]:
### A COMPLÉTER ###
# Objectif : lancer le Run 2 dans la même expérience MLflow que le Run 1
#
# 1. Utiliser mlflow.set_experiment("fastia-llama-finetuning") — même expérience
# 2. Nommer le run avec RUN2_CONFIG["run_name"]
# 3. Loguer tous les paramètres de RUN2_CONFIG
#    Ajouter également : dataset_path, dataset_size, max_length
# 4. Lancer l'entraînement
# 5. Loguer train_loss et eval_loss
#
# Le output_dir doit être different du Run 1 : ./model_run2

mlflow.set_experiment("fastia-llama-finetuning")

with mlflow.start_run(run_name=RUN2_CONFIG["run_name"]):
    mlflow.log_params({
        "model_id": MODEL_ID,
        "lora_r": RUN2_CONFIG["lora_r"],
        "lora_alpha": RUN2_CONFIG["lora_alpha"],
        "lora_dropout": RUN2_CONFIG["lora_dropout"],
        "learning_rate": RUN2_CONFIG["learning_rate"],
        "num_train_epochs": RUN2_CONFIG["num_train_epochs"],
        "batch_size": RUN2_CONFIG["per_device_train_batch_size"],
        "dataset_path": dataset_path,
        "dataset-size": len(data),
        "max_length": MAX_LENGTH,
    })

    training_args = TrainingArguments(
        output_dir="./model_run2",
        num_train_epochs=RUN2_CONFIG["num_train_epochs"],
        per_device_train_batch_size=RUN2_CONFIG["per_device_train_batch_size"],
        learning_rate=RUN2_CONFIG["learning_rate"],
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=tokenizer
    )

    result = trainer.train()

    mlflow.log_metric("train_loss", result.training_loss)

    # Retirer le callback lié au Notebook ---
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)

    eval_result = trainer.evaluate()
    mlflow.log_metric("eval_loss", eval_result["eval_loss"])

    print(f"Train loss : {result.training_loss:.4f}")
    print(f"Eval loss  : {eval_result['eval_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Epoch,Training Loss,Validation Loss
1,1.530521,1.225273
2,0.640808,0.515769
3,0.343551,0.328256
4,0.301355,0.316942


Train loss : 1.1112
Eval loss  : 0.3169


---
## 7. Comparaison Run 1 vs Run 2

Les deux runs sont dans la même expérience MLflow.
On peut maintenant les comparer directement.

In [19]:
# Récupération des deux runs depuis MLflow
runs = mlflow.search_runs(
    experiment_names=["fastia-llama-finetuning"],
    order_by=["start_time ASC"]
)

colonnes = [
    "tags.mlflow.runName",
    "params.learning_rate",
    "params.num_train_epochs",
    "params.dataset_size",
    "params.lora_r",
    "params.lora_dropout",
    "metrics.train_loss",
    "metrics.eval_loss"
]

print(runs[colonnes].to_string(index=False))

tags.mlflow.runName params.learning_rate params.num_train_epochs params.dataset_size params.lora_r params.lora_dropout  metrics.train_loss  metrics.eval_loss
 run1-lr2e4-3epochs               0.0002                       3                  86             8                0.05            1.756628           0.593054
               run2               0.0002                       4                None             8                0.05            1.111182           0.316942


In [ ]:
### A COMPLÉTER ###
# Objectif : comparer les prédictions qualitatives des deux runs
#
# 1. Définir la fonction predict() — identique au Brief 1
#
# 2. Charger le modèle du Run 1 depuis ./model_run1
#    Indice : PeftModel.from_pretrained(base_model, "./model_run1")
#
# 3. Charger le modèle du Run 2 depuis ./model_run2
#
# 4. Sur les mêmes 5 demandes problématiques identifiées dans le Brief 1,
#    comparer les prédictions des deux runs côte à côte :
#    Demande | Prédiction Run 1 | Prédiction Run 2
#
# 5. Les erreurs observées en Brief 1 ont-elles été corrigées ?
#    Répondez dans un commentaire.

def predict(texte, model, tokenizer, max_new_tokens=150):
    prompt = (
        f"<s>[INST] Rôle : Tu es un expert en classification de tickets support pour FastIA."
        f"Mission : Analyse la demande utilisateur et renvoie exclusivement un objet JSON."
        f"Contraintes strictes :"
        f"Format : Réponds uniquement en JSON pur, sans texte avant ou après (pas de \"Voici le résultat\")."
        f"Champs obligatoires : categorie, priorite, reponse_suggeree."
        f"Catégories autorisées (Choisir exclusivement parmi) : "
        f"Support technique"
        f"Demande commerciale"
        f"Demande de transformation"
        f"Réclamation"
        f"Information générale"
        f"Priorités autorisées : normale, haute."
        f"Langue : Tout le contenu du JSON doit être en Français.\n\n"
        f"Demande : {texte} [/INST]"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # --- AJOUT : Nettoyage de la sortie ---
    # 1. On retire le token de fin de séquence résiduel
    generated = generated.replace("</s>", "")
    # 2. On retire les éventuelles balises Markdown
    generated = generated.replace("```json", "").replace("```", "")
    # 3. On enlève les espaces ou sauts de ligne restants
    generated = generated.strip()
        
    try:
        return json.loads(generated)
    except json.JSONDecodeError:
        return {"erreur": "JSON invalide", "brut": generated}


# Chargement des deux modèles
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "./model_run1/checkpoint-51", adapter_name="run1")
model.load_adapter("./model_run2/checkpoint-100", adapter_name="run2")
# model_run2 = PeftModel.from_pretrained(base_model, "./model_run2")

# Demandes problématiques du Brief 1
demandes_problematiques = [
    "Mon ordinateur ne s'allume plus et fait un bruit bizarre.", # Support technique
    "J'aimerais obtenir un devis pour l'achat de 10 nouvelles licences.", # Demande commerciale
    "Pouvez-vous migrer mon compte actuel vers l'offre Entreprise ?", # Demande de transformation
    "Je suis très mécontent, mon colis est arrivé cassé !", # Réclamation
    "Quels sont vos tarifs pour le support premium ?", # Information générale
    "Le logiciel plante dès que je clique sur le bouton imprimer.", # Support technique
    "Je souhaite résilier mon abonnement et passer chez un concurrent.", # Réclamation
    "Est-il possible d'ajouter une option stockage cloud à mon contrat ?", # Demande de transformation
    "Je voudrais parler à un commercial pour un partenariat.", # Demande commerciale
    "Avez-vous une documentation sur l'utilisation de l'API ?" # Information générale
]

for demande in demandes_problematiques:
    model.set_adapter("run1")
    pred_run1 = predict(demande, model, tokenizer)
    model.set_adapter("run2")
    pred_run2 = predict(demande, model, tokenizer)
    print(f"Demande : {demande}")
    print(f"Run 1   : {pred_run1}")
    print(f"Run 2   : {pred_run2}")
    print("---")

# Votre analyse :
# Les erreurs observées en Brief 1 ont-elles été corrigées ?
#   Oui, le modèle génère désormais des JSON valides cependant les catégories/priorités ne sont pas toujours correctes.

Loading weights: 100%|██████████| 146/146 [00:02<00:00, 61.12it/s]


Demande : Mon ordinateur ne s'allume plus et fait un bruit bizarre.
Run 1   : {'erreur': 'JSON invalide', 'brut': ''}
Run 2   : {'categorie': 'Information générale', 'priorite': 'normale', 'reponse_suggeree': 'Veuillez confirmer que votre ordinateur est en panne et que vous avez un problème technique.'}
---
Demande : J'aimerais obtenir un devis pour l'achat de 10 nouvelles licences.
Run 1   : {'erreur': 'JSON invalide', 'brut': ''}
Run 2   : {'categorie': 'Demande commerciale', 'priorite': 'normale', 'reponse_suggeree': 'Nous pouvons vous proposer un devis personnalisé pour votre nouvelle installation. Nous vous contacterons dès que possible.'}
---
Demande : Pouvez-vous migrer mon compte actuel vers l'offre Entreprise ?
Run 1   : {'erreur': 'JSON invalide', 'brut': 'Rôle : Expert en support technique.Mission : Analyse la demande utilisateur et renvoie exclusivement un objet JSON.Contraintes strictes :Format : Réponds uniquement en JSON pur, sans texte avant ou après (pas de "Voici le r

---
## 8. Choix et sauvegarde du meilleur modèle

Le choix du meilleur modèle ne se fait pas uniquement sur la loss.
Une eval_loss légèrement plus haute peut être acceptable si les prédictions
qualitatives sont meilleures sur les cas problématiques.

In [ ]:
### A COMPLÉTER ###
# Objectif : justifier le choix du meilleur run et sauvegarder le modèle
#
# 1. Indiquer quel run vous choisissez : Run 1 ou Run 2
#
# 2. Justifier ce choix en vous appuyant sur :
#    - Les métriques MLflow (train_loss, eval_loss)
#    - La comparaison qualitative des prédictions
#    - L'impact de votre action corrective
#
# 3. Sauvegarder le meilleur modèle dans ./model_final
#    Indice : model.save_pretrained("./model_final")
#             tokenizer.save_pretrained("./model_final")
#
# 4. Vérifier le contenu du dossier sauvegardé

MEILLEUR_RUN = "run2"
SAVE_PATH = "./model_final"

# Justification :
# Réponse du modèle du Run2 avec un bon formatage JSON

# Sauvegarde
meilleur_model = PeftModel.from_pretrained(base_model, f"./model_run2/checkpoint-100", adapter_name=MEILLEUR_RUN)
meilleur_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)


print(f"Modèle sauvegardé dans {SAVE_PATH}")

Tokenizer chargé


Loading weights: 100%|██████████| 146/146 [00:01<00:00, 73.53it/s]


Modèle sauvegardé dans ./model_final


---
## 9. Bilan de l'itération

Ce bilan sera repris dans le README.md du Brief 2.

In [ ]:
### A COMPLÉTER ###
# Objectif : produire un bilan structuré de l'itération
#
# Répondez aux questions suivantes dans des commentaires :
#
# 1. L'action corrective a-t-elle eu l'effet attendu ?
#    Comparez les métriques et prédictions avant / après.
#
# 2. Y a-t-il des cas problématiques qui persistent malgré la correction ?
#    Si oui, quelle en est la cause probable ?
#
# 3. Si vous deviez faire un Run 3, quelle action corrective appliqueriez-vous ?
#    (Cette question est ouverte, il n'y a pas de Run 3 dans ce brief)
#
# 4. Le modèle final est-il prêt à être exposé via une API ?
#    Quels sont ses points forts et ses limites connues ?

# 1. Effet de l'action corrective :
#   L'action corrective a permis d'obtenir des JSON valides, 
#   ce qui était un problème majeur du Run 1. Cependant, les catégories et priorités ne sont pas toujours correctes, 
#   indiquant que le modèle a encore des difficultés à les definir.

# 2. Cas persistants :
#   Oui, les erreurs de classification des catégories et priorités persistent.

# 3. Run 3 hypothétique :
#   Ajouter encore plus d'exemples ciblés pour renforcer la compréhension des catégories et priorités.


# 4. Prêt pour la production :
#  Le modèle n'est pas encore prêt pour une exposition en production.

---
## Récapitulatif

Etapes réalisées dans ce notebook :

1. Relecture et synthèse du diagnostic du Brief 1
2. Mise en oeuvre de l'action corrective (dataset et/ou hyperparamètres)
3. Préparation du dataset pour le Run 2
4. Configuration LoRA avec les paramètres ajustés
5. Entraînement Run 2 tracké dans MLflow
6. Comparaison Run 1 vs Run 2 : métriques et prédictions qualitatives
7. Choix et sauvegarde du meilleur modèle
8. Bilan de l'itération

Prochaine étape — Brief 3 : conteneurisation avec Docker, logging Loguru,
tests Pytest et documentation de déploiement.